# 01 — Event Loop et Coroutines

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- expliquer le modèle d'exécution **single-thread asynchrone** ;
- écrire des coroutines avec `async def` et `await` ;
- comprendre le rôle de l'**event loop** et son fonctionnement ;
- utiliser `asyncio.run()` et `asyncio.sleep()` ;
- distinguer coroutine, coroutine function et awaitable.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le threading, `Lock`, `Event`, le GIL ;
- `multiprocessing` et `concurrent.futures` ;
- les générateurs et l'instruction `yield` ;
- les gestionnaires de contexte (`with`) ;
- les décorateurs.

Notions que nous allons **introduire** ici :

- `async def`, `await`, coroutines natives ;
- la boucle événementielle (`event loop`) ;
- `asyncio.run()`, `asyncio.sleep()`.

## Plan

1. Synchrone vs asynchrone : le modèle mental
2. `async def` — définir une coroutine
3. `await` — suspendre l'exécution
4. `asyncio.run()` — lancer la boucle
5. L'event loop en détail
6. `asyncio.sleep()` vs `time.sleep()`
7. Coroutine, awaitable, future : le vocabulaire
8. Exécuter asyncio dans Jupyter
9. Synthèse
10. Exercices

---

## 1. Synchrone vs asynchrone : le modèle mental

### Le problème

En programmation synchrone, quand votre code fait une requête réseau ou lit un fichier, le thread **attend** (bloque) jusqu'à la réponse. Si vous avez 100 requêtes à faire, chacune prenant 100ms, le total est ~10 secondes.

### La solution asynchrone

Au lieu de bloquer, le code **suspend** son exécution et rend la main à l'event loop, qui peut exécuter d'autres tâches en attendant. Quand la réponse arrive, l'exécution reprend.

| Modèle | Threads | Mémoire | Complexité | Idéal pour |
|---|---|---|---|---|
| Synchrone | 1 thread par tâche | Élevée | Faible | Scripts simples |
| Threading | N threads | Moyenne | Moyenne | I/O modéré |
| **Asyncio** | **1 seul thread** | **Faible** | Moyenne | I/O massif (milliers de connexions) |

---

## 2. `async def` — définir une coroutine

Une **coroutine function** est déclarée avec `async def`. Elle retourne un objet **coroutine** qui doit être `await`-é pour s'exécuter.

In [ ]:
async def bonjour() -> str:
    return "Bonjour depuis une coroutine !"

# Appeler la fonction retourne un objet coroutine, pas le résultat
coro = bonjour()
print(f"Type : {type(coro)}")
print(f"Repr : {coro}")

# Il faut await pour obtenir le résultat
resultat = await coro
print(f"Résultat : {resultat}")

**Note Jupyter :** dans un notebook, la boucle asyncio est déjà en cours d'exécution. On peut donc utiliser `await` directement. Dans un script Python normal, il faut utiliser `asyncio.run()` (voir section 4).

In [ ]:
import inspect

async def ma_coro() -> int:
    return 42

print(f"Est une coroutine function : {inspect.iscoroutinefunction(ma_coro)}")
print(f"Est une coroutine : {inspect.iscoroutine(ma_coro())}")

---

## 3. `await` — suspendre l'exécution

`await` **suspend** la coroutine courante et rend la main à l'event loop. Quand l'opération attendue est terminée, l'exécution reprend après le `await`.

In [ ]:
import asyncio

async def tache(nom: str, duree: float) -> str:
    print(f"[{nom}] Début")
    await asyncio.sleep(duree)  # suspend, rend la main
    print(f"[{nom}] Fin après {duree}s")
    return f"{nom} terminée"

resultat = await tache("T1", 0.5)
print(resultat)

### Ce qu'on peut `await`

On peut `await` tout objet **awaitable** :

- Une coroutine (`async def`)
- Un `asyncio.Task`
- Un `asyncio.Future`
- Tout objet avec `__await__`

In [ ]:
import asyncio

async def etape1() -> int:
    await asyncio.sleep(0.1)
    return 10

async def etape2(x: int) -> int:
    await asyncio.sleep(0.1)
    return x * 2

async def pipeline() -> int:
    a = await etape1()
    b = await etape2(a)
    return b

resultat = await pipeline()
print(f"Pipeline : {resultat}")

---

## 4. `asyncio.run()` — lancer la boucle

Dans un script Python normal (pas Jupyter), on utilise `asyncio.run()` pour créer l'event loop, exécuter une coroutine, et fermer la boucle.

```python
# script.py
import asyncio

async def main() -> None:
    print("Hello, asyncio !")
    await asyncio.sleep(1)
    print("Done.")

asyncio.run(main())
```

In [ ]:
import asyncio

# Dans Jupyter, la boucle est déjà en cours
loop = asyncio.get_running_loop()
print(f"Event loop en cours : {loop}")
print(f"Loop est en train de tourner : {loop.is_running()}")

### `asyncio.run()` vs boucle Jupyter

| Contexte | Comment exécuter |
|---|---|
| Script `.py` | `asyncio.run(main())` |
| Jupyter / IPython | `await main()` directement |
| REPL Python 3.14+ | `python -m asyncio` puis `await` |

---

## 5. L'event loop en détail

L'event loop est le **coeur** d'asyncio. Son fonctionnement simplifié :

```
while tasks_pending:
    event = poll_for_ready_events()  # I/O, timers
    for callback in ready_callbacks:
        callback()  # reprend les coroutines suspendues
```

C'est un modèle **coopératif** : chaque coroutine doit rendre la main volontairement avec `await`. Si une coroutine fait un calcul long sans `await`, elle **bloque toutes les autres**.

In [ ]:
import asyncio
import time

async def cooperative(nom: str) -> None:
    for i in range(3):
        print(f"[{nom}] étape {i}")
        await asyncio.sleep(0.1)  # rend la main

# Les deux coroutines s'entrelacent
await asyncio.gather(
    cooperative("A"),
    cooperative("B"),
)

In [ ]:
import asyncio
import time

async def bloquante(nom: str) -> None:
    print(f"[{nom}] calcul bloquant...")
    time.sleep(1)  # BLOQUE l'event loop ! Pas de await.
    print(f"[{nom}] fini")

async def gentille(nom: str) -> None:
    print(f"[{nom}] attend...")
    await asyncio.sleep(0.1)
    print(f"[{nom}] OK")

start = time.perf_counter()
await asyncio.gather(bloquante("lente"), gentille("rapide"))
print(f"Total : {time.perf_counter() - start:.2f}s")
print("'rapide' est retardée car 'lente' bloque l'event loop !")

---

## 6. `asyncio.sleep()` vs `time.sleep()`

| Fonction | Bloque le thread ? | Rend la main ? |
|---|---|---|
| `time.sleep(1)` | **Oui** | Non |
| `await asyncio.sleep(1)` | Non | **Oui** |

In [ ]:
import asyncio
import time

async def demo_sleep() -> None:
    print("Avec asyncio.sleep (non bloquant) :")
    start = time.perf_counter()
    await asyncio.gather(
        asyncio.sleep(1),
        asyncio.sleep(1),
        asyncio.sleep(1),
    )
    print(f"  3 x asyncio.sleep(1) = {time.perf_counter() - start:.2f}s (≈1s)")

await demo_sleep()

### Exécuter du code bloquant sans bloquer l'event loop

Pour les opérations bloquantes inévitables (bibliothèques non-async), utiliser `run_in_executor()` :

In [ ]:
import asyncio
import time

def bloquant() -> str:
    time.sleep(1)  # bloquant
    return "résultat bloquant"

async def main() -> None:
    loop = asyncio.get_running_loop()
    # Exécute dans un thread séparé
    resultat = await loop.run_in_executor(None, bloquant)
    print(resultat)

await main()

---

## 7. Coroutine, awaitable, future : le vocabulaire

| Terme | Définition |
|---|---|
| **Coroutine function** | Fonction déclarée avec `async def` |
| **Coroutine** | Objet retourné par un appel à une coroutine function |
| **Awaitable** | Tout objet qui peut être `await`-é |
| **Task** | Coroutine enveloppée et planifiée sur l'event loop |
| **Future** | Promesse de résultat futur (bas niveau) |

In [ ]:
import asyncio
import inspect

async def exemple() -> int:
    return 42

coro = exemple()

print(f"iscoroutinefunction(exemple) : {inspect.iscoroutinefunction(exemple)}")
print(f"iscoroutine(coro)            : {inspect.iscoroutine(coro)}")
print(f"isawaitable(coro)            : {inspect.isawaitable(coro)}")

# Il faut await la coroutine sinon elle est jamais exécutée
result = await coro
print(f"Résultat : {result}")

### Awaitable personnalisé avec `__await__`

In [ ]:
import asyncio

class MonAwaitable:
    def __init__(self, valeur: int) -> None:
        self.valeur = valeur

    def __await__(self):
        yield  # cède le contrôle une fois
        return self.valeur * 2

async def utiliser() -> None:
    result = await MonAwaitable(21)
    print(f"Résultat : {result}")

await utiliser()

---

## 8. Exécuter asyncio dans Jupyter

Jupyter utilise `nest_asyncio` ou un event loop intégré (depuis IPython 7+). Les cellules `await` fonctionnent directement.

### Attention aux pièges

- `asyncio.run()` lève une erreur dans Jupyter (loop déjà en cours).
- Utilisez `await` directement ou `asyncio.ensure_future()`.

In [ ]:
import asyncio

# Ceci fonctionne dans Jupyter
async def hello() -> str:
    await asyncio.sleep(0.1)
    return "Hello from async!"

print(await hello())

In [ ]:
import asyncio

# Ceci lèverait une erreur dans Jupyter :
# asyncio.run(hello())  # RuntimeError: This event loop is already running

# Alternative : créer une tâche
task = asyncio.ensure_future(hello())
await task
print(f"Via ensure_future : {task.result()}")

---

## 9. Synthèse

| Concept | Clé |
|---|---|
| `async def fn()` | Déclare une coroutine function |
| `await expr` | Suspend et attend un awaitable |
| `asyncio.run(coro)` | Point d'entrée dans un script |
| `asyncio.sleep(n)` | Sleep non bloquant |
| Event loop | Exécute les coroutines en coopération |
| `run_in_executor()` | Exécute du code bloquant dans un thread |
| Awaitable | Tout objet avec `__await__` |

**Règle d'or :** ne jamais bloquer l'event loop. Si vous devez appeler du code synchrone bloquant, utilisez `run_in_executor()`.

**Quand utiliser asyncio vs threading ?**

- Milliers de connexions simultanées → asyncio
- Quelques tâches I/O → threading suffit
- CPU-bound → multiprocessing (ou free-threaded)

---

## 10. Exercices

### Exercice 1 — Première coroutine *(facile)*

Écrire une coroutine `saluer(nom: str, delai: float)` qui attend `delai` secondes puis retourne `"Bonjour, {nom} !"`. L'appeler pour 3 noms **séquentiellement** (un await après l'autre), puis mesurer le temps.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Event_loop_et_coroutines", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

async def saluer(nom: str, delai: float) -> str:
    await asyncio.sleep(delai)
    return f"Bonjour, {nom} !"

start = time.perf_counter()
r1 = await saluer("Alice", 0.3)
r2 = await saluer("Bob", 0.3)
r3 = await saluer("Charlie", 0.3)
elapsed = time.perf_counter() - start

print(r1, r2, r3)
print(f"Temps : {elapsed:.2f}s (séquentiel ≈ 0.9s)")
```

</details>

### Exercice 2 — Parallèle avec `gather` *(facile)*

Reprendre l'exercice 1 mais exécuter les 3 appels en parallèle avec `asyncio.gather()`. Mesurer le temps et constater la différence.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Event_loop_et_coroutines", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

async def saluer(nom: str, delai: float) -> str:
    await asyncio.sleep(delai)
    return f"Bonjour, {nom} !"

start = time.perf_counter()
resultats = await asyncio.gather(
    saluer("Alice", 0.3),
    saluer("Bob", 0.3),
    saluer("Charlie", 0.3),
)
elapsed = time.perf_counter() - start

for r in resultats:
    print(r)
print(f"Temps : {elapsed:.2f}s (parallèle ≈ 0.3s)")
```

</details>

### Exercice 3 — Compteur asynchrone *(moyen)*

Écrire deux coroutines :

- `compter(nom, n, delai)` : affiche les nombres de 1 à n avec un `asyncio.sleep(delai)` entre chaque.

Lancer deux compteurs en parallèle (`gather`) avec des délais différents (0.1s et 0.2s). Observer l'entrelacement.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Event_loop_et_coroutines", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio

async def compter(nom: str, n: int, delai: float) -> None:
    for i in range(1, n + 1):
        print(f"[{nom}] {i}")
        await asyncio.sleep(delai)

await asyncio.gather(
    compter("Rapide", 10, 0.1),
    compter("Lent", 5, 0.2),
)
print("Les deux compteurs sont terminés.")
```

</details>

### Exercice 4 — Intégrer du code bloquant *(moyen)*

Écrire une fonction synchrone `travail_lourd(n: int) -> int` qui fait `sum(range(n))`. L'appeler depuis une coroutine via `run_in_executor()`, en parallèle de 3 `asyncio.sleep(0.5)`. Vérifier que tout finit en ~0.5s (pas 2s).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Event_loop_et_coroutines", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

def travail_lourd(n: int) -> int:
    return sum(range(n))

async def main() -> None:
    loop = asyncio.get_running_loop()
    start = time.perf_counter()

    resultat, *_ = await asyncio.gather(
        loop.run_in_executor(None, travail_lourd, 10_000_000),
        asyncio.sleep(0.5),
        asyncio.sleep(0.5),
        asyncio.sleep(0.5),
    )

    elapsed = time.perf_counter() - start
    print(f"Résultat : {resultat}")
    print(f"Temps : {elapsed:.2f}s (≈0.5s, pas 2s)")

await main()
```

</details>

### Exercice 5 — Timer asynchrone réutilisable *(difficile)*

Écrire une classe `AsyncTimer` qui :

- S'utilise comme context manager async : `async with AsyncTimer() as t:`
- Mesure le temps écoulé (via `__aenter__` et `__aexit__`)
- Expose `t.elapsed` après la sortie du bloc

Tester avec un `asyncio.sleep(1)` dans le bloc.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Event_loop_et_coroutines", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

class AsyncTimer:
    def __init__(self) -> None:
        self.elapsed: float = 0.0
        self._start: float = 0.0

    async def __aenter__(self) -> 'AsyncTimer':
        self._start = time.perf_counter()
        return self

    async def __aexit__(self, *exc) -> None:
        self.elapsed = time.perf_counter() - self._start

async with AsyncTimer() as t:
    await asyncio.sleep(1)

print(f"Temps mesuré : {t.elapsed:.2f}s")
```

</details>

---

## Ressources

- [docs Python — `asyncio`](https://docs.python.org/3/library/asyncio.html)
- [RealPython — Async IO in Python](https://realpython.com/async-io-python/)
- *Fluent Python* (L. Ramalho), chapitres 21-22
- [PEP 492 — Coroutines with async and await syntax](https://peps.python.org/pep-0492/)